# 🤖 F1 Qualifying Position Prediction - Baseline Model

**Goal:** Build a baseline ML model to predict qualifying positions

**Pipeline:**
1. Data preparation and train/test split
2. Feature engineering and selection
3. Model training (Random Forest, XGBoost, LightGBM)
4. Model evaluation and comparison
5. Prediction analysis
6. Error analysis
7. Model interpretation

In [1]:
# Imports
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

# Try to import XGBoost
try:
    import xgboost as xgb
    XGBOOST_AVAILABLE = True
    print("✅ XGBoost available")
except ImportError as e:
    XGBOOST_AVAILABLE = False
    print("⚠️  XGBoost not available")
    print(f"   Error: {str(e)[:100]}...")
    print("   Install with: brew install libomp && pip install xgboost")

# Try to import LightGBM
try:
    import lightgbm as lgb
    LIGHTGBM_AVAILABLE = True
    print("✅ LightGBM available")
except ImportError:
    LIGHTGBM_AVAILABLE = False
    print("⚠️  LightGBM not available")
    print("   Install with: pip install lightgbm")


import warnings
warnings.filterwarnings('ignore')

print("✅ Imports successful")

✅ XGBoost available
✅ LightGBM available
✅ Imports successful


---
## 1️⃣ Load and Prepare Data

In [2]:
# Load features
df = pd.read_parquet('../data/features/ml_features_2022_2025.parquet')

# Focus on qualifying
df_qual = df[df['qualifying_position'].notna()].copy()

print(f"📦 Dataset shape: {df_qual.shape}")
print(f"📅 Years: {df_qual['year'].min()} - {df_qual['year'].max()}")

# Define target
target_col = 'qualifying_position'

📦 Dataset shape: (1778, 55)
📅 Years: 2022 - 2025


In [3]:
# Define features (using top features from importance analysis)
# Select top features from each category

top_features = [
    # Historical (strongest predictors)
    'circuit_avg_position', 'circuit_best_position',
    'recent_avg_position', 'recent_best_position', 'form_trend',
    'team_circuit_avg_position', 'team_momentum',
    
    # Telemetry
    'max_throttle_ratio', 'brake_max_g', 'braking_events',
    
    # Weather
    'avg_rainfall', 'avg_track_temp',
    
    # Tire
    'tyre_age', 'is_fresh_tyre',
    
    # Weather performance
    'wet_dry_delta'
]

# Filter to existing columns
available_features = [f for f in top_features if f in df_qual.columns]

print(f"\n📊 Using {len(available_features)} features:")
for feat in available_features:
    print(f"   - {feat}")


📊 Using 12 features:
   - circuit_avg_position
   - circuit_best_position
   - recent_avg_position
   - recent_best_position
   - form_trend
   - team_circuit_avg_position
   - team_momentum
   - max_throttle_ratio
   - brake_max_g
   - avg_rainfall
   - avg_track_temp
   - wet_dry_delta


In [4]:
# Prepare modeling dataset
df_model = df_qual[available_features + [target_col, 'year', 'event', 'driver']].copy()

# Simple median imputation for missing values
for col in available_features:
    if df_model[col].isnull().any():
        median_val = df_model[col].median()
        df_model[col].fillna(median_val, inplace=True)
        print(f"   Imputed {col}: {df_model[col].isnull().sum()} missing")

print(f"\n✅ Model dataset: {df_model.shape}")
print(f"✅ Missing values: {df_model[available_features].isnull().sum().sum()}")

   Imputed circuit_avg_position: 0 missing
   Imputed circuit_best_position: 0 missing
   Imputed recent_avg_position: 0 missing
   Imputed recent_best_position: 0 missing
   Imputed form_trend: 0 missing
   Imputed team_circuit_avg_position: 0 missing
   Imputed team_momentum: 0 missing
   Imputed wet_dry_delta: 0 missing

✅ Model dataset: (1778, 16)
✅ Missing values: 0


---
## 2️⃣ Train/Test Split Strategy

In [5]:
# Time-based split (train on earlier years, test on latest year)
# This is more realistic than random split!

train_years = [2022, 2023, 2024]
test_year = 2025

train_df = df_model[df_model['year'].isin(train_years)].copy()
test_df = df_model[df_model['year'] == test_year].copy()

X_train = train_df[available_features]
y_train = train_df[target_col]

X_test = test_df[available_features]
y_test = test_df[target_col]

print(f"📊 Split summary:")
print(f"   Train: {len(X_train):,} samples from {train_years}")
print(f"   Test:  {len(X_test):,} samples from {test_year}")
print(f"   Split ratio: {len(X_train)/(len(X_train)+len(X_test)):.1%} train")

📊 Split summary:
   Train: 1,359 samples from [2022, 2023, 2024]
   Test:  419 samples from 2025
   Split ratio: 76.4% train


---
## 3️⃣ Model Training

In [6]:
# Train multiple models
models = {}
predictions = {}
metrics = []

print("🤖 Training models...\n")

🤖 Training models...



In [7]:
# Model 1: Random Forest
print("🌲 Random Forest...")

rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

models['Random Forest'] = rf_model
predictions['Random Forest'] = rf_pred

# Metrics
rf_mae = mean_absolute_error(y_test, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_r2 = r2_score(y_test, rf_pred)

metrics.append({
    'Model': 'Random Forest',
    'MAE': rf_mae,
    'RMSE': rf_rmse,
    'R²': rf_r2
})

print(f"   MAE: {rf_mae:.3f} positions")
print(f"   RMSE: {rf_rmse:.3f}")
print(f"   R²: {rf_r2:.3f}\n")

🌲 Random Forest...
   MAE: 3.883 positions
   RMSE: 4.832
   R²: 0.295



In [8]:
# Model 2: XGBoost (if available)
if XGBOOST_AVAILABLE:
    print("🚀 XGBoost...")
    
    xgb_model = xgb.XGBRegressor(
        n_estimators=200,
        max_depth=8,
        learning_rate=0.05,
        min_child_weight=3,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    )
    
    xgb_model.fit(X_train, y_train)
    xgb_pred = xgb_model.predict(X_test)
    
    models['XGBoost'] = xgb_model
    predictions['XGBoost'] = xgb_pred
    
    xgb_mae = mean_absolute_error(y_test, xgb_pred)
    xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_pred))
    xgb_r2 = r2_score(y_test, xgb_pred)
    
    metrics.append({
        'Model': 'XGBoost',
        'MAE': xgb_mae,
        'RMSE': xgb_rmse,
        'R²': xgb_r2
    })
    
    print(f"   MAE: {xgb_mae:.3f} positions")
    print(f"   RMSE: {xgb_rmse:.3f}")
    print(f"   R²: {xgb_r2:.3f}\n")

🚀 XGBoost...
   MAE: 3.779 positions
   RMSE: 4.742
   R²: 0.321



In [9]:
# Model 3: LightGBM (if available)
if LIGHTGBM_AVAILABLE:
    print("⚡ LightGBM...")
    
    lgb_model = lgb.LGBMRegressor(
        n_estimators=200,
        max_depth=10,
        learning_rate=0.05,
        num_leaves=31,
        min_child_samples=20,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )
    
    lgb_model.fit(X_train, y_train)
    lgb_pred = lgb_model.predict(X_test)
    
    models['LightGBM'] = lgb_model
    predictions['LightGBM'] = lgb_pred
    
    lgb_mae = mean_absolute_error(y_test, lgb_pred)
    lgb_rmse = np.sqrt(mean_squared_error(y_test, lgb_pred))
    lgb_r2 = r2_score(y_test, lgb_pred)
    
    metrics.append({
        'Model': 'LightGBM',
        'MAE': lgb_mae,
        'RMSE': lgb_rmse,
        'R²': lgb_r2
    })
    
    print(f"   MAE: {lgb_mae:.3f} positions")
    print(f"   RMSE: {lgb_rmse:.3f}")
    print(f"   R²: {lgb_r2:.3f}\n")

⚡ LightGBM...
   MAE: 3.777 positions
   RMSE: 4.695
   R²: 0.334



---
## 4️⃣ Model Comparison

In [10]:
# Metrics comparison table
metrics_df = pd.DataFrame(metrics)

print("📊 Model Performance Comparison:\n")
print(metrics_df.to_string(index=False))

# Find best model
best_model_name = metrics_df.loc[metrics_df['MAE'].idxmin(), 'Model']
print(f"\n🏆 Best Model: {best_model_name}")

📊 Model Performance Comparison:

        Model      MAE     RMSE       R²
Random Forest 3.882942 4.831560 0.295031
      XGBoost 3.778611 4.741886 0.320956
     LightGBM 3.777412 4.694609 0.334429

🏆 Best Model: LightGBM


In [11]:
# Visualize model comparison
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=('Mean Absolute Error', 'RMSE', 'R² Score')
)

# MAE
fig.add_trace(
    go.Bar(x=metrics_df['Model'], y=metrics_df['MAE'], name='MAE', marker_color='indianred'),
    row=1, col=1
)

# RMSE
fig.add_trace(
    go.Bar(x=metrics_df['Model'], y=metrics_df['RMSE'], name='RMSE', marker_color='lightseagreen'),
    row=1, col=2
)

# R²
fig.add_trace(
    go.Bar(x=metrics_df['Model'], y=metrics_df['R²'], name='R²', marker_color='royalblue'),
    row=1, col=3
)

fig.update_layout(
    title_text='🏆 Model Performance Comparison',
    showlegend=False,
    height=400
)

fig.update_yaxes(title_text='MAE (positions)', row=1, col=1)
fig.update_yaxes(title_text='RMSE', row=1, col=2)
fig.update_yaxes(title_text='R²', row=1, col=3)

fig.show()

---
## 5️⃣ Prediction Analysis

In [12]:
# Use best model for detailed analysis
best_model = models[best_model_name]
best_predictions = predictions[best_model_name]

# Create results dataframe
results_df = test_df[['year', 'event', 'driver']].copy()
results_df['actual'] = y_test.values
results_df['predicted'] = best_predictions
results_df['error'] = results_df['predicted'] - results_df['actual']
results_df['abs_error'] = np.abs(results_df['error'])

print(f"📊 Prediction Statistics ({best_model_name}):\n")
print(f"   Mean Absolute Error: {results_df['abs_error'].mean():.3f} positions")
print(f"   Median Absolute Error: {results_df['abs_error'].median():.3f} positions")
print(f"   Max Error: {results_df['abs_error'].max():.3f} positions")
print(f"   Std Dev of Error: {results_df['error'].std():.3f}")

📊 Prediction Statistics (LightGBM):

   Mean Absolute Error: 3.777 positions
   Median Absolute Error: 3.100 positions
   Max Error: 13.191 positions
   Std Dev of Error: 4.700


In [13]:
# Actual vs Predicted scatter plot
fig = px.scatter(
    results_df,
    x='actual',
    y='predicted',
    hover_data=['driver', 'event'],
    title=f'🎯 Actual vs Predicted Qualifying Position ({best_model_name})',
    labels={'actual': 'Actual Position', 'predicted': 'Predicted Position'},
    opacity=0.6,
    height=600
)

# Add perfect prediction line
fig.add_trace(
    go.Scatter(
        x=[1, 20],
        y=[1, 20],
        mode='lines',
        name='Perfect Prediction',
        line=dict(color='red', dash='dash')
    )
)

fig.update_layout(
    xaxis=dict(range=[0, 21]),
    yaxis=dict(range=[0, 21])
)

fig.show()

In [14]:
# Error distribution
fig = px.histogram(
    results_df,
    x='error',
    nbins=40,
    title='📊 Prediction Error Distribution',
    labels={'error': 'Error (Predicted - Actual)'},
    height=500
)

fig.add_vline(x=0, line_dash="dash", line_color="red", annotation_text="Zero Error")
fig.show()

print(f"\n📈 Error Distribution:")
print(f"   Over-predictions (positive error): {(results_df['error'] > 0).sum()} ({100*(results_df['error'] > 0).mean():.1f}%)")
print(f"   Under-predictions (negative error): {(results_df['error'] < 0).sum()} ({100*(results_df['error'] < 0).mean():.1f}%)")
print(f"   Within ±1 position: {(results_df['abs_error'] <= 1).sum()} ({100*(results_df['abs_error'] <= 1).mean():.1f}%)")
print(f"   Within ±2 positions: {(results_df['abs_error'] <= 2).sum()} ({100*(results_df['abs_error'] <= 2).mean():.1f}%)")
print(f"   Within ±3 positions: {(results_df['abs_error'] <= 3).sum()} ({100*(results_df['abs_error'] <= 3).mean():.1f}%)")


📈 Error Distribution:
   Over-predictions (positive error): 218 (52.0%)
   Under-predictions (negative error): 201 (48.0%)
   Within ±1 position: 59 (14.1%)
   Within ±2 positions: 128 (30.5%)
   Within ±3 positions: 199 (47.5%)


---
## 6️⃣ Error Analysis

In [15]:
# Best and worst predictions
print("🏆 Best Predictions (smallest error):\n")
print(results_df.nsmallest(10, 'abs_error')[['driver', 'event', 'actual', 'predicted', 'abs_error']].to_string(index=False))

print("\n\n❌ Worst Predictions (largest error):\n")
print(results_df.nlargest(10, 'abs_error')[['driver', 'event', 'actual', 'predicted', 'abs_error']].to_string(index=False))

🏆 Best Predictions (smallest error):

driver                     event  actual  predicted  abs_error
   PIA       Canadian Grand Prix     3.0   3.001970   0.001970
   DOO     Australian Grand Prix    14.0  14.028552   0.028552
   SAI        British Grand Prix    11.0  10.947594   0.052406
   LAW       Japanese Grand Prix    14.0  13.944526   0.055474
   COL Emilia Romagna Grand Prix    15.0  15.070535   0.070535
   HAD      Singapore Grand Prix     8.0   8.112567   0.112567
   VER        Chinese Grand Prix     4.0   4.147306   0.147306
   HAD        Spanish Grand Prix     9.0   8.842583   0.157417
   SAI  Saudi Arabian Grand Prix     6.0   6.167517   0.167517
   RUS          Dutch Grand Prix     5.0   5.185697   0.185697


❌ Worst Predictions (largest error):

driver                  event  actual  predicted  abs_error
   GAS    Canadian Grand Prix    20.0   6.809368  13.190632
   GAS     Italian Grand Prix    19.0   5.918182  13.081818
   SAI    Austrian Grand Prix    19.0   6.033521 

In [16]:
# Error by driver
error_by_driver = results_df.groupby('driver')['abs_error'].agg(['mean', 'count']).reset_index()
error_by_driver = error_by_driver[error_by_driver['count'] >= 3]  # Min 3 predictions
error_by_driver = error_by_driver.sort_values('mean')

fig = px.bar(
    error_by_driver,
    x='mean',
    y='driver',
    orientation='h',
    title='📊 Average Prediction Error by Driver (min 3 sessions)',
    labels={'mean': 'Mean Absolute Error', 'driver': 'Driver'},
    color='mean',
    color_continuous_scale='RdYlGn_r',
    height=600
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

In [17]:
# Error by actual position
error_by_position = results_df.groupby('actual')['abs_error'].mean().reset_index()

fig = px.bar(
    error_by_position,
    x='actual',
    y='abs_error',
    title='📊 Prediction Error by Actual Position',
    labels={'actual': 'Actual Qualifying Position', 'abs_error': 'Mean Absolute Error'},
    height=500
)
fig.show()

print("\n💡 Analysis:")
print("   - Model may be more accurate for certain grid positions")
print("   - Midfield often harder to predict than top/bottom teams")


💡 Analysis:
   - Model may be more accurate for certain grid positions
   - Midfield often harder to predict than top/bottom teams


In [18]:
df['baseline_pred'] = df['recent_avg_position']
baseline_mae = (df['qualifying_position'] - df['baseline_pred']).abs().mean()
print(f"Baseline MAE (just using recent_avg): {baseline_mae:.2f}")
print(f"Your model MAE: {results_df['abs_error'].mean():.3f}")
print(f"Improvement: {baseline_mae - results_df['abs_error'].mean():.2f} positions")

Baseline MAE (just using recent_avg): 3.60
Your model MAE: 3.777
Improvement: -0.18 positions


In [19]:
df['baseline_pred'] = df['recent_avg_position']
baseline_mae = (df['qualifying_position'] - df['baseline_pred']).abs().mean()
improvement = baseline_mae - results_df['abs_error'].mean()
print(f"Baseline MAE (just using recent_avg): {baseline_mae:.2f}")
print(f"Model MAE: {results_df['abs_error'].mean():.3f}")
print(f"Improvement: {improvement:.2f} positions")

if improvement > 0:
    import joblib
    import os
    import json

    # Go one level up from notebook’s working directory
    base_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))

    model_dir = os.path.join(base_dir, 'models')
    os.makedirs(model_dir, exist_ok=True)

    model_path = os.path.join(model_dir, 'quali_model.pkl')
    joblib.dump(best_model, model_path)

    print(f"✅ Saved quali model to {model_path}")

    # Save metadata next to it
    metadata = {
        'features': available_features,
        'model_name': best_model_name,
        'mae': metrics_df.loc[metrics_df['Model'] == best_model_name, 'MAE'].values[0],
        'r2': metrics_df.loc[metrics_df['Model'] == best_model_name, 'R²'].values[0]
    }

    metadata_path = os.path.join(model_dir, 'quali_model_metadata.json')
    with open(metadata_path, 'w') as f:
        json.dump(metadata, f, indent=2)

    print(f"✅ Saved quali model metadata to {metadata_path}")
else:
    print("⚠️  Model did not improve over baseline, not saving.")


Baseline MAE (just using recent_avg): 3.60
Model MAE: 3.777
Improvement: -0.18 positions
⚠️  Model did not improve over baseline, not saving.
